<a href="https://colab.research.google.com/github/Rohit-0612/HHGoa26-Voice-Rag/blob/main/notebooks/colab_backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HH Goa 2026 — Voice-RAG backend on ColabRuns the FastAPI backend and exposes it publicly through a Cloudflare tunnel.**Why Colab:** the backend needs ~2.3GB RAM (the jina reranker is ~1.3GB of it and is*not* optional — its score is the working off-topic guardrail). Every 512MB free PaaStier OOMs. Colab gives 12–13GB.**Known limitation:** Colab disconnects after ~90 min idle and hard-stops at 12h.Re-run this notebook to get a fresh URL, then paste it into the frontend's"Backend URL" field — no frontend redeploy needed.

In [16]:
import os

print("Current directory:")
!pwd

print("\n/content/app contents:")
!ls -la /content/app

print("\nGit status:")
!git -C /content/app status 2>&1 || true

Current directory:
/content/app

/content/app contents:
total 132
drwxr-xr-x 10 root root  4096 Aug 22 15:48 .
drwxr-xr-x  1 root root  4096 Aug 22 15:48 ..
drwxr-xr-x  3 root root  4096 Aug 22 15:48 data
-rw-r--r--  1 root root 56398 Aug 22 15:48 decision.md
-rw-r--r--  1 root root  1730 Aug 22 15:48 Dockerfile
-rw-r--r--  1 root root   159 Aug 22 15:48 .dockerignore
-rw-r--r--  1 root root   800 Aug 22 15:48 .env.example
drwxr-xr-x  8 root root  4096 Aug 22 15:48 .git
drwxr-xr-x  3 root root  4096 Aug 22 15:48 .github
-rw-r--r--  1 root root   307 Aug 22 15:48 .gitignore
drwxr-xr-x  2 root root  4096 Aug 22 15:48 notebooks
-rw-r--r--  1 root root   874 Aug 22 15:48 pyproject.toml
-rw-r--r--  1 root root 14177 Aug 22 15:48 README.md
drwxr-xr-x  2 root root  4096 Aug 22 15:48 scripts
drwxr-xr-x  8 root root  4096 Aug 22 15:48 src
drwxr-xr-x  2 root root  4096 Aug 22 15:48 tests
drwxr-xr-x  2 root root  4096 Aug 22 15:48 web

Git status:
On branch main
Your branch is up to date with 'or

In [17]:
!pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.1/324.1 kB 16.8 MB/s eta 0:00:00
  Building editable for hhgoa26-voice-rag (pyproject.toml) ... done


In [18]:
!python -c "import sys; print(sys.version)"

3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]


## 1. Clone + install (~3 min)

In [19]:
!git clone https://github.com/Rohit-0612/HHGoa26-Voice-Rag.git /content/app
%cd /content/app
!pip install -q -r requirements.txt

fatal: destination path '/content/app' already exists and is not an empty directory.
/content/app
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## 2. SecretsAdd these in Colab's **🔑 Secrets** panel (left sidebar), each with notebook access on:`QDRANT_URL`, `QDRANT_API_KEY`, `GROQ_API_KEY`, `SARVAM_API_KEY`, `NIM_API_KEY`

In [20]:
from google.colab import userdata
import pathlib

KEYS = [
    "QDRANT_URL",
    "QDRANT_API_KEY",
    "GROQ_API_KEY",
    "SARVAM_API_KEY",
    "NIM_API_KEY",
]

lines = []
missing = []

for k in KEYS:
    try:
        v = userdata.get(k)
        if v:
            lines.append(f"{k}={v}")
        else:
            missing.append(k)
    except Exception:
        missing.append(k)

# Non-secret configuration
lines += [
    "QDRANT_COLLECTION=msmarco_xi",
    "GROQ_MODEL=openai/gpt-oss-20b",
    "MAX_TOKENS=1024",
    "DENSE_MODEL=sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "SPARSE_MODEL=Qdrant/bm25",
    "RERANK_MODEL=jinaai/jina-reranker-v2-base-multilingual",
    "NIM_MODEL=nvidia/nvidia-nemotron-nano-9b-v2",
    "CORS_ORIGINS=*",
]

pathlib.Path("/content/app/.env").write_text(
    "\n".join(lines) + "\n"
)

print("wrote .env with", len(lines), "settings")

if missing:
    print("MISSING (add in the Secrets panel):", missing)
else:
    print("all secrets present")

wrote .env with 13 settings
all secrets present


## 3. Pre-download models (~2 min)\n\nDoing this now means the first real request is fast.

In [21]:
from fastembed import TextEmbedding, SparseTextEmbeddingfrom fastembed.rerank.cross_encoder import TextCrossEncoderTextEmbedding('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')SparseTextEmbedding('Qdrant/bm25')TextCrossEncoder('jinaai/jina-reranker-v2-base-multilingual')print("models cached")

SyntaxError: invalid syntax (1296524757.py, line 1)

## 4. Start server + public tunnel

In [ ]:
import subprocess, time, re, os, threading!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64!chmod +x /usr/local/bin/cloudflaredos.chdir("/content/app")server = subprocess.Popen(    ["python","-m","uvicorn","src.api.main:app","--host","0.0.0.0","--port","8000"],    stdout=open("/content/server.log","w"), stderr=subprocess.STDOUT)# Wait for model load before opening the tunnel.for _ in range(120):    time.sleep(3)    if "Application startup complete" in open("/content/server.log").read():        print("backend ready"); breakelse:    print("backend slow to start; check /content/server.log")tunnel = subprocess.Popen(    ["cloudflared","tunnel","--url","http://localhost:8000","--no-autoupdate"],    stdout=open("/content/tunnel.log","w"), stderr=subprocess.STDOUT)url = Nonefor _ in range(40):    time.sleep(2)    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/tunnel.log").read())    if m: url = m.group(0); breakprint("\n" + "="*62)print("PUBLIC BACKEND URL:", url or "NOT FOUND -- see /content/tunnel.log")print("Paste this into the frontend's 'Backend URL' field.")print("="*62)

## 5. Verify

In [ ]:
import requestsr = requests.get(f"{url}/health", timeout=60)print(r.status_code, r.json())

## 6. Keep alive\n\nRun this cell and leave the tab open; it pings /health so Colab does not idle out.

In [ ]:
import time, requestswhile True:    try:        requests.get(f"{url}/health", timeout=30)        print(".", end="", flush=True)    except Exception as e:        print("x", end="", flush=True)    time.sleep(300)